In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import os
import sys
from tqdm.contrib.concurrent import process_map  # multiprocessing + tqdm
from functools import partial
import multiprocessing

from concurrent.futures import ProcessPoolExecutor, as_completed
sys.path.append("../../../src/irl")
import reddit_traj_construction



/data/fryuan/anaconda3/envs/irl_py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
with open("./config.json") as f:
    config = json.load(f)
    collection_start_date = int(config["collection_start_date"])
    collection_end_date = int(config["collection_end_date"])
    root_output_dir_path = config["root_output_dir_path"]
    deberta_weight_path = config["deberta_weight_path"]
    
    processed_per_user_path =  root_output_dir_path + "/4_per_user_processing_collated" 


load_dir = processed_per_user_path
output_dir = root_output_dir_path + "/user_content"
os.makedirs(output_dir, exist_ok=True)

all_users = sorted(os.listdir(load_dir))
print(len(all_users))


22845


In [ ]:
# rus active timeframe 
start_date = "2015-01-02"
end_date = "2018-04-11"

# Helper function to run in parallel
def get_user_df(user):
    df_slice = reddit_traj_construction.get_user_activity_in_timeframe(
        user,
        load_dir,
        start_date=start_date,
        end_date=end_date
    )
    # Only keep relevant rows and columns
    filtered = df_slice[df_slice.action.isin([1, 2, 3])].copy()

    # Ensure column exists
    if 'agree_prediction' not in filtered.columns:
        filtered['agree_prediction'] = None
    if 'url' not in filtered.columns:
        filtered['url'] = None
    if 'title' not in filtered.columns:
        filtered['title'] = None
    if 'selftext' not in filtered.columns:
        filtered['selftext'] = None
    if 'subreddit' not in filtered.columns:
        filtered['subreddit'] = None

    return filtered[["author", "id", "created_utc", "subreddit", "body", "selftext", "action", "url", "title", 'agree_prediction']].copy()

# Use all available cores
num_workers = multiprocessing.cpu_count()

df_list = []
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(get_user_df, user): user for user in all_users}
    
    for counter, future in enumerate(as_completed(futures), 1):
        try:
            df = future.result()
            df_list.append(df)
            print(counter, "/", len(all_users))
        except Exception as e:
            print(f"Error processing user {futures[future]}: {e}")

# Concatenate all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_pickle(output_dir + "/all_user_active_content_df")

1 / 22845
2 / 22845
3 / 22845
4 / 22845
5 / 22845
6 / 22845
7 / 22845
8 / 22845
9 / 22845
10 / 22845
11 / 22845
12 / 22845
13 / 22845
14 / 22845
15 / 22845
16 / 22845
17 / 22845
18 / 22845
19 / 22845
20 / 22845
21 / 22845
22 / 22845
23 / 22845
24 / 22845
25 / 22845
26 / 22845
27 / 22845
28 / 22845
29 / 22845
30 / 22845
31 / 22845
32 / 22845
33 / 22845
34 / 22845
35 / 22845
36 / 22845
37 / 22845
38 / 22845
39 / 22845
40 / 22845
41 / 22845
42 / 22845
43 / 22845
44 / 22845
45 / 22845
46 / 22845
47 / 22845
48 / 22845
49 / 22845
50 / 22845
51 / 22845
52 / 22845
53 / 22845
54 / 22845
55 / 22845
56 / 22845
57 / 22845
58 / 22845
59 / 22845
60 / 22845
61 / 22845
62 / 22845
63 / 22845
64 / 22845
65 / 22845
66 / 22845
67 / 22845
68 / 22845
69 / 22845
70 / 22845
71 / 22845
72 / 22845
73 / 22845
74 / 22845
75 / 22845
76 / 22845
77 / 22845
78 / 22845
79 / 22845
80 / 22845
81 / 22845
82 / 22845
83 / 22845
84 / 22845
85 / 22845
86 / 22845
87 / 22845
88 / 22845
89 / 22845
90 / 22845
91 / 22845
92 / 228

In [ ]:
# content df
content = combined_df[combined_df.action.isin([1, 2])]
content.to_pickle(output_dir +"/content_text_df.pkl")
# submission df
submissions = combined_df[combined_df.action == 1]
submissions['submission_text'] = np.where(
    submissions['selftext'].isna() | (submissions['selftext'] == "[removed]"),
    submissions['title'],
    submissions['title'] + " " + submissions['selftext']
)
submissions.to_pickle(output_dir +"/submission_text_df.pkl")


In [ ]:
# get passive actions (replies and parents)

# rus active timeframe
start_date = "2015-01-02"
end_date = "2018-04-11"

# Helper function to run in parallel
def get_user_df(user):
    df_slice = reddit_traj_construction.get_user_activity_in_timeframe(
        user,
        load_dir,
        start_date=start_date,
        end_date=end_date
    )
    # Only keep relevant rows and columns
    filtered = df_slice[df_slice.action.isin([4,5])].copy()

    # Ensure column exists
    if 'agree_prediction' not in filtered.columns:
        filtered['agree_prediction'] = None
    if 'url' not in filtered.columns:
        filtered['url'] = None
    if 'title' not in filtered.columns:
        filtered['title'] = None
    if 'selftext' not in filtered.columns:
        filtered['selftext'] = None
    if 'subreddit' not in filtered.columns:
        filtered['subreddit'] = None

    return filtered[["author", "id", "created_utc", "subreddit", "body", "selftext", "action", "url", "title", 'agree_prediction']].copy()

# Use all available cores
num_workers = multiprocessing.cpu_count()

df_list = []
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(get_user_df, user): user for user in all_users}
    
    for counter, future in enumerate(as_completed(futures), 1):
        try:
            df = future.result()
            df_list.append(df)
            print(counter, "/", len(all_users))
        except Exception as e:
            print(f"Error processing user {futures[future]}: {e}")

# Concatenate all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_pickle(output_dir + "/all_user_passive_content_df")

1 / 22845
2 / 22845
3 / 22845
4 / 22845
5 / 22845
6 / 22845
7 / 22845
8 / 22845
9 / 22845
10 / 22845
11 / 22845
12 / 22845
13 / 22845
14 / 22845
15 / 22845
16 / 22845
17 / 22845
18 / 22845
19 / 22845
20 / 22845
21 / 22845
22 / 22845
23 / 22845
24 / 22845
25 / 22845
26 / 22845
27 / 22845
28 / 22845
29 / 22845
30 / 22845
31 / 22845
32 / 22845
33 / 22845
34 / 22845
35 / 22845
36 / 22845
37 / 22845
38 / 22845
39 / 22845
40 / 22845
41 / 22845
42 / 22845
43 / 22845
44 / 22845
45 / 22845
46 / 22845
47 / 22845
48 / 22845
49 / 22845
50 / 22845
51 / 22845
52 / 22845
53 / 22845
54 / 22845
55 / 22845
56 / 22845
57 / 22845
58 / 22845
59 / 22845
60 / 22845
61 / 22845
62 / 22845
63 / 22845
64 / 22845
65 / 22845
66 / 22845
67 / 22845
68 / 22845
69 / 22845
70 / 22845
71 / 22845
72 / 22845
73 / 22845
74 / 22845
75 / 22845
76 / 22845
77 / 22845
78 / 22845
79 / 22845
80 / 22845
81 / 22845
82 / 22845
83 / 22845
84 / 22845
85 / 22845
86 / 22845
87 / 22845
88 / 22845
89 / 22845
90 / 22845
91 / 22845
92 / 228